# 1 Missing Values Analysis
Comprehensive analysis of missing Final_Lat and Final_Long values in the
final dataset. Explores patterns by year, state, chain, address type, and
data source availability.

Input: data/1_derived/5_geocode_truck_stops/7_final_truck_stops.csv

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
IN_FILE = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops' / '7_final_truck_stops.csv'

df = pd.read_csv(IN_FILE, low_memory=False)
print(f'Loaded {len(df):,} rows')

In [ ]:
# 1. Missing values overview
total = len(df)
has_lat = df['Final_Lat'].notna()
has_long = df['Final_Long'].notna()
has_both = has_lat & has_long
missing_both = ~has_lat & ~has_long
lat_only = has_lat & ~has_long
long_only = ~has_lat & has_long

print('=== Missing Values Summary ===')
print(f'Total rows       : {total:,}')
print(f'Has both coords  : {has_both.sum():,} ({has_both.sum()/total*100:.1f}%)')
print(f'Missing both     : {missing_both.sum():,} ({missing_both.sum()/total*100:.1f}%)')
print(f'Lat only         : {lat_only.sum():,}')
print(f'Long only        : {long_only.sum():,}')

# Pie chart
fig, ax = plt.subplots(figsize=(8, 6))
sizes = [has_both.sum(), missing_both.sum(), lat_only.sum() + long_only.sum()]
labels = ['Complete', 'Missing Both', 'Partial']
ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
ax.set_title('Coordinate Completeness')
plt.show()

In [ ]:
# 2. Breakdown by year
if 'year' in df.columns:
    by_year = df.groupby('year').apply(
        lambda g: pd.Series({
            'total': len(g),
            'complete': (g['Final_Lat'].notna() & g['Final_Long'].notna()).sum(),
            'missing': (g['Final_Lat'].isna() | g['Final_Long'].isna()).sum()
        })
    )
    by_year['pct_complete'] = (by_year['complete'] / by_year['total'] * 100).round(1)
    print('=== Completion by Year ===')
    print(by_year.to_string())

    fig, ax = plt.subplots(figsize=(12, 5))
    by_year[['complete', 'missing']].plot(kind='bar', stacked=True, ax=ax)
    ax.set_title('Coordinate Completeness by Year')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# 3. Breakdown by state
if 'state' in df.columns:
    by_state = df.groupby('state').apply(
        lambda g: pd.Series({
            'total': len(g),
            'complete': (g['Final_Lat'].notna() & g['Final_Long'].notna()).sum()
        })
    )
    by_state['pct'] = (by_state['complete'] / by_state['total'] * 100).round(1)
    by_state = by_state.sort_values('pct')
    print('=== Completion by State (sorted by %) ===')
    print(by_state.to_string())

In [ ]:
# 4. Breakdown by chain
if 'chain' in df.columns:
    by_chain = df.groupby('chain').apply(
        lambda g: pd.Series({
            'total': len(g),
            'complete': (g['Final_Lat'].notna() & g['Final_Long'].notna()).sum()
        })
    )
    by_chain['pct'] = (by_chain['complete'] / by_chain['total'] * 100).round(1)
    by_chain = by_chain.sort_values('total', ascending=False)
    print('=== Completion by Chain (top 20 by count) ===')
    print(by_chain.head(20).to_string())

In [ ]:
# 5. Breakdown by Address_Type
if 'Address_Type' in df.columns:
    by_type = df.groupby('Address_Type').apply(
        lambda g: pd.Series({
            'total': len(g),
            'complete': (g['Final_Lat'].notna() & g['Final_Long'].notna()).sum()
        })
    )
    by_type['pct'] = (by_type['complete'] / by_type['total'] * 100).round(1)
    print('=== Completion by Address Type ===')
    print(by_type.to_string())

In [ ]:
# 6. Data source availability for missing records
missing = df[~has_both].copy()
complete_df = df[has_both].copy()

print('=== Data Source Comparison: Complete vs Missing ===')
for col in ['Scraped_phone_match_rate', 'Yelp_phone_match_rate', 'Yellowbook_phone_match_rate']:
    if col in df.columns:
        pct_complete = (complete_df[col] == True).mean() * 100
        pct_missing = (missing[col] == True).mean() * 100 if len(missing) > 0 else 0
        print(f'{col}:')
        print(f'  Complete records: {pct_complete:.1f}%')
        print(f'  Missing records : {pct_missing:.1f}%')
        print()